In [ ]:
# Mount Google Drive to access input/output data
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:
# ============================================================
# Imports, Configuration, and Dataset Load
# ============================================================
# Loads a NOAA CRW 5km SST/DHW NetCDF file using xarray.
# Update CRW_PATH to point to your downloaded NetCDF file.
# Files can be obtained from the NOAA CoralTemp data portal:
#   https://coralreefwatch.noaa.gov/product/5km/index.php
# ============================================================

import xarray as xr
import numpy as np
import rioxarray
import os

# --- User configuration ---
# Path to the NOAA CRW NetCDF file
CRW_PATH   = "/content/drive/MyDrive/<YOUR_PROJECT_FOLDER>/NOAA CRW/Data/<YOUR_CRW_FILE>.nc"

# Directory where output GeoTIFFs will be saved
OUTPUT_DIR = "/content/drive/MyDrive/<YOUR_PROJECT_FOLDER>/NOAA CRW/Data/"

# Study region bounding box (decimal degrees)
LAT_MIN, LAT_MAX =  15.7,  18.5
LON_MIN, LON_MAX = -89.5, -87.3

# Time window for composite (inclusive)
TIME_START = "2023-05-01"
TIME_END   = "2023-11-30"

# Load the CRW dataset
ds = xr.open_dataset(CRW_PATH)


In [ ]:
# ============================================================
# Subset to Study Region and Time Window
# ============================================================
# Selects the CRW_SST variable over the bounding box and date
# range defined in the configuration cell.
#
# Note: CRW latitude coordinates are stored in descending order
# (north to south), so the slice is passed as (lat_max, lat_min).
# ============================================================

sst = ds['CRW_SST'].sel(
    time=slice(TIME_START, TIME_END),
    latitude=slice(LAT_MAX, LAT_MIN),
    longitude=slice(LON_MIN, LON_MAX)
)


In [ ]:
# ============================================================
# Compute Composites and Save as GeoTIFFs
# ============================================================
# Produces two composites from the subsetted SST stack:
#
#   1. Mean composite — pixel-wise temporal mean across all
#      time steps, ignoring NaN (cloud/missing) values.
#
#   2. Valid pixel count — number of non-NaN observations
#      per pixel across the time window. Useful as a data
#      availability / confidence layer alongside the mean.
#
# Both outputs are saved as tiled, LZW-compressed GeoTIFFs
# in WGS84 (EPSG:4326). The count raster is saved as int16
# since pixel counts are whole numbers.
#
# rioxarray requires spatial dims to be declared explicitly
# before writing because xarray uses named dimensions
# ('latitude'/'longitude') rather than the CF convention
# ('y'/'x') that rioxarray expects by default.
# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)


def prepare_spatial(da):
    """
    Assign spatial dimension names and CRS to an xarray DataArray
    so rioxarray can write it as a georeferenced GeoTIFF.

    Parameters
    ----------
    da : xr.DataArray - 2-D DataArray with 'longitude' and 'latitude' coords.

    Returns
    -------
    xr.DataArray - Same array with spatial dims and EPSG:4326 CRS attached.
    """
    da = da.rio.set_spatial_dims(x_dim='longitude', y_dim='latitude')
    da = da.rio.write_crs("EPSG:4326")
    return da


# --- 1. Mean composite ---
mean_composite = sst.mean(dim='time', skipna=True)
mean_composite = prepare_spatial(mean_composite)
mean_composite.rio.to_raster(
    os.path.join(OUTPUT_DIR, "NOAA_CRW_SST_Mean_Composite.tif"),
    tiled=True,
    compress='LZW'
)

# --- 2. Valid pixel count ---
valid_count = sst.count(dim='time')
valid_count = prepare_spatial(valid_count)
valid_count.rio.to_raster(
    os.path.join(OUTPUT_DIR, "NOAA_CRW_SST_ValidPixelCount.tif"),
    tiled=True,
    compress='LZW',
    dtype='int16'
)
